# Prithvi-EO-2.0-300M Accuracy Improvement: Linear Probing & Biophysical Gating

**Master Thesis:** *Catching Greenwashing from Space: Verifying Palm Oil Zero-Deforestation Claims Using Satellite Data and NLP*
**Author:** Ritik Ghoghari (GISMA University of Applied Sciences, Berlin)
**Supervisor:** Professor Mohamad Hoseini

---

## Motivation: Why This Notebook Exists
In our initial batch evaluation across 290 mills, the zero-shot frozen Prithvi-EO-2.0-300M foundation model yielded a network mean **Spearman $\rho = 0.172$** and **IoU = 0.086**.

**The Root Cause:**
Prithvi was pretrained as a self-supervised Masked Autoencoder (MAE) to reconstruct missing image patches. It was never trained on deforestation labels. Unsupervised cosine distance flags *any* spectral difference (soil moisture, road construction, crop harvesting, sun angle, and cloud haze) rather than actual tree canopy loss.

## The Solution Implemented in this Notebook
We test and compare three distinct paradigms on the high-loss **IOI / SYARIMO** benchmark mill:
1. **Baseline:** Unsupervised embedding cosine distance (the original POC).
2. **Technique 2 (Instant Zero-Training Filter):** Biophysical $\Delta\text{NDVI}$ Gating (rejecting changes where green biomass did not crash) combined with Otsu dynamic thresholding.
3. **Technique 1 (AI Gold Standard):** Supervised Linear Probing on frozen Prithvi embeddings. We keep the 300M parameter backbone frozen and train a lightweight 2-layer MLP classification head on Hansen post-2020 loss labels.

---
### Quick Instructions for Google Colab:
1. **Runtime > Change runtime type > T4 GPU** (free tier).
2. **Runtime > Run all.**
3. Click through the Google Drive mount prompt (or let Earth Engine fallback fetch if files are not yet in Drive).
4. The entire notebook finishes in ~5 to 8 minutes on a T4 GPU.

### Cell 1: Install Dependencies
**Note:** Installs `numpy<2.0` and `terratorch`. Because Google Colab preloads NumPy 2.x in memory, **you must click `Runtime > Restart session` once after running this cell**.

In [ ]:
!pip install -q "numpy<2.0" terratorch rasterio earthengine-api scikit-learn scikit-image matplotlib scipy pandas tifffile
print("\n" + "="*70)
print("✅ Dependencies installed successfully!")
print("⚠️ ACTION REQUIRED: In the top menu, click 'Runtime > Restart session' (or press Ctrl+M .)")
print("   Then run from Cell 2 onward. (Do NOT re-run Cell 1).")
print("="*70)

### Cell 2: Mount Google Drive & Locate Imagery

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
POC_DIR = '/content/drive/MyDrive/thesis_prithvi_poc'

FILES = {
    'before': os.path.join(POC_DIR, 'sentinel2_before_2019.tif'),
    'after': os.path.join(POC_DIR, 'sentinel2_after_2024.tif'),
    'hansen': os.path.join(POC_DIR, 'hansen_groundtruth.tif')
}

MISSING = [k for k, v in FILES.items() if not os.path.exists(v)]
if MISSING:
    print(f"Files missing in Drive: {MISSING}. Will pull via Earth Engine fallback in Cell 3.")
else:
    print("All 3 GeoTIFF files found in Google Drive! Ready to load.")

### Cell 3: Earth Engine Fallback (Runs only if Drive files are missing)

In [ ]:
if MISSING:
    import ee
    import urllib.request
    ee.Authenticate()
    ee.Initialize(project='thesis-greenwashing')

    # IOI / SYARIMO Coordinates
    LAT, LON = 5.334037, 117.781334
    BUFFER_M = 10000
    BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
    point = ee.Geometry.Point([LON, LAT])
    region = point.buffer(BUFFER_M).bounds()

    def mask_clouds(img):
        scl = img.select('SCL')
        mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
        return img.updateMask(mask)

    def get_s2_composite(start, end):
        coll = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterDate(start, end).filterBounds(region)
                .map(mask_clouds).select(BANDS))
        return coll.median().multiply(0.0001).clip(region)

    def download_tif(image, scale, path):
        url = image.getDownloadURL({
            'region': region, 'scale': scale, 'format': 'GEO_TIFF',
            'crs': 'EPSG:4326', 'maxPixels': 1e10
        })
        urllib.request.urlretrieve(url, path)
        print(f"Downloaded -> {path}")

    os.makedirs('/content/prithvi_fallback', exist_ok=True)
    if 'before' in MISSING:
        FILES['before'] = '/content/prithvi_fallback/sentinel2_before_2019.tif'
        download_tif(get_s2_composite('2019-01-01', '2019-12-31'), 10, FILES['before'])
    if 'after' in MISSING:
        FILES['after'] = '/content/prithvi_fallback/sentinel2_after_2024.tif'
        download_tif(get_s2_composite('2023-06-01', '2024-12-31'), 10, FILES['after'])
    if 'hansen' in MISSING:
        gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
        hansen_ref = (gfc.select('treecover2000').gt(30).rename('forest2000')
                      .addBands(gfc.select('lossyear').gte(21).rename('loss_post2020')).clip(region))
        FILES['hansen'] = '/content/prithvi_fallback/hansen_groundtruth.tif'
        download_tif(hansen_ref, 30, FILES['hansen'])
else:
    print("Drive files verified. No download needed.")

### Cell 4: Load & Crop GeoTIFFs to Common Grid

In [ ]:
import rasterio
import numpy as np

def read_geotiff(path):
    with rasterio.open(path) as src:
        arr = src.read()
    return np.nan_to_num(arr, nan=0.0)

before_arr = np.clip(read_geotiff(FILES['before']), 0, 1)  # (6, H, W)
after_arr = np.clip(read_geotiff(FILES['after']), 0, 1)    # (6, H, W)
hansen_arr = read_geotiff(FILES['hansen'])                 # (2, Hh, Wh)

H = min(before_arr.shape[1], after_arr.shape[1])
W = min(before_arr.shape[2], after_arr.shape[2])
before_arr = before_arr[:, :H, :W]
after_arr = after_arr[:, :H, :W]

# Resample Hansen (30m) to Sentinel-2 (10m) grid (3x repeat)
loss_30m = hansen_arr[1]
hansen_loss_10m = np.kron(loss_30m, np.ones((3, 3)))[:H, :W]
if hansen_loss_10m.shape[0] < H or hansen_loss_10m.shape[1] < W:
    pad_h = H - hansen_loss_10m.shape[0]
    pad_w = W - hansen_loss_10m.shape[1]
    hansen_loss_10m = np.pad(hansen_loss_10m, ((0, pad_h), (0, pad_w)), mode='edge')

print(f"Aligned S2 and Hansen grids to common dimensions: {H} x {W} pixels at 10m")

### Cell 5: Load Pretrained Prithvi-EO-2.0-300M Encoder

In [ ]:
import torch
from terratorch.registry import BACKBONE_REGISTRY

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

CANDIDATES = ["prithvi_eo_v2_300", "prithvi_eo_v2_300m", "ibm-nasa-geospatial/Prithvi-EO-2.0-300M"]
model = None
for cand in CANDIDATES:
    try:
        model = BACKBONE_REGISTRY.build(cand, pretrained=True)
        print(f"Successfully loaded Prithvi backbone using key: '{cand}'")
        break
    except Exception as e:
        continue

if model is None:
    raise RuntimeError("Failed to load Prithvi model from registry. Check terratorch installation.")

# Freeze the 300M parameter backbone
for p in model.parameters():
    p.requires_grad = False
model = model.to(device).eval()
print("Backbone frozen successfully. Embeddings will be extracted with zero gradient overhead.")

### Cell 6: Tile Extraction & Feature Embedding Extraction
Extracts both tile-level embeddings and biophysical NDVI features.

In [ ]:
TILE = 224
n_tiles_y = H // TILE
n_tiles_x = W // TILE
n_total = n_tiles_y * n_tiles_x
print(f"Tiling into {n_tiles_y} x {n_tiles_x} = {n_total} tiles of {TILE}px")

def extract_embedding(tile_tensor):
    # tile_tensor: (1, 6, 1, 224, 224)
    with torch.no_grad():
        out = model(tile_tensor)
    feat = out[-1] if isinstance(out, (list, tuple)) else out
    if feat.dim() == 3:       # (B, N_tokens, C)
        feat = feat.mean(dim=1)
    elif feat.dim() > 2:
        feat = feat.flatten(2).mean(dim=2)
    return feat.squeeze(0).cpu()   # (768,)

# Pre-allocate containers
emb_before_list = []
emb_after_list = []
hansen_fraction_list = []
d_ndvi_tile_list = []

# Calculate full-resolution NDVI (Band 8 = index 3, Band 4 = index 2)
ndvi_before = (before_arr[3] - before_arr[2]) / (before_arr[3] + before_arr[2] + 1e-6)
ndvi_after = (after_arr[3] - after_arr[2]) / (after_arr[3] + after_arr[2] + 1e-6)
d_ndvi_full = ndvi_after - ndvi_before  # Negative values indicate vegetation loss

for ty in range(n_tiles_y):
    for tx in range(n_tiles_x):
        y0, x0 = ty * TILE, tx * TILE
        b_chip = before_arr[:, y0:y0 + TILE, x0:x0 + TILE]
        a_chip = after_arr[:, y0:y0 + TILE, x0:x0 + TILE]
        loss_chip = hansen_loss_10m[y0:y0 + TILE, x0:x0 + TILE]
        d_ndvi_chip = d_ndvi_full[y0:y0 + TILE, x0:x0 + TILE]

        # Normalize using combined tile statistics
        comb = np.stack([b_chip, a_chip], axis=0)
        m = comb.mean(axis=(0, 2, 3), keepdims=True).squeeze(0)
        s = comb.std(axis=(0, 2, 3), keepdims=True).squeeze(0) + 1e-6
        b_norm = torch.from_numpy((b_chip - m) / s).float().unsqueeze(0).unsqueeze(2).to(device)
        a_norm = torch.from_numpy((a_chip - m) / s).float().unsqueeze(0).unsqueeze(2).to(device)

        e_b = extract_embedding(b_norm)
        e_a = extract_embedding(a_norm)

        emb_before_list.append(e_b)
        emb_after_list.append(e_a)
        hansen_fraction_list.append(loss_chip.mean())
        d_ndvi_tile_list.append(d_ndvi_chip.mean())

X_before = torch.stack(emb_before_list)        # (n_tiles, 768)
X_after = torch.stack(emb_after_list)          # (n_tiles, 768)
y_loss_frac = np.array(hansen_fraction_list)   # (n_tiles,)
d_ndvi_tiles = np.array(d_ndvi_tile_list)      # (n_tiles,)

print(f"Extracted {n_total} tile embeddings (Dim: {X_before.shape[1]})")

### Cell 7: Experiment 1 — Baseline Unsupervised Cosine Distance
Evaluates the unmodified baseline POC method.

In [ ]:
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import precision_score, recall_score, jaccard_score

# Unsupervised Cosine Dissimilarity (1 - cos_sim)
cos_sim = F.cosine_similarity(X_before, X_after, dim=1).numpy()
baseline_change = 1.0 - cos_sim

# Ground Truth Binary Mask (>30% of tile lost post-2020)
y_binary = (y_loss_frac > 0.30).astype(int)
target_rate = y_binary.mean()

# Baseline percentile threshold
thresh_base = np.quantile(baseline_change, 1 - target_rate)
pred_baseline = (baseline_change >= thresh_base).astype(int)

rho_base, p_base = spearmanr(baseline_change, y_loss_frac)
prec_base = precision_score(y_binary, pred_baseline, zero_division=0)
rec_base = recall_score(y_binary, pred_baseline, zero_division=0)
iou_base = jaccard_score(y_binary, pred_baseline, zero_division=0)

print("=== EXPERIMENT 1: BASELINE UNSUPERVISED COSINE ===")
print(f"Spearman Rho : {rho_base:.3f} (p={p_base:.4f})")
print(f"Precision    : {prec_base:.3f}")
print(f"Recall       : {rec_base:.3f}")
print(f"IoU (Jaccard): {iou_base:.3f}")

### Cell 8: Experiment 2 — Technique 2: Biophysical $\Delta\text{NDVI}$ Gating & Dynamic Otsu
Enforces a vegetation loss constraint ($\Delta\text{NDVI} < -0.10$) and dynamically thresholds via Otsu's method.

In [ ]:
from skimage.filters import threshold_otsu

# Combine Cosine Dissimilarity with Vegetation Loss Gating
# If dNDVI is positive (vegetation increased or unchanged), suppress score to 0
veg_drop_weight = np.clip(-d_ndvi_tiles, 0, 1.0)
gated_change = baseline_change * veg_drop_weight

# Dynamic Otsu thresholding on gated signal
otsu_val = threshold_otsu(gated_change)
pred_gated = (gated_change >= otsu_val).astype(int)

rho_gated, p_gated = spearmanr(gated_change, y_loss_frac)
prec_gated = precision_score(y_binary, pred_gated, zero_division=0)
rec_gated = recall_score(y_binary, pred_gated, zero_division=0)
iou_gated = jaccard_score(y_binary, pred_gated, zero_division=0)

print("=== EXPERIMENT 2: BIOPHYSICAL dNDVI GATING + OTSU ===")
print(f"Spearman Rho : {rho_gated:.3f} (p={p_gated:.4f})")
print(f"Precision    : {prec_gated:.3f}")
print(f"Recall       : {rec_gated:.3f}")
print(f"IoU (Jaccard): {iou_gated:.3f}")

### Cell 9: Experiment 3 — Technique 1: Supervised Linear Probing Head
Trains a lightweight 2-layer MLP classification head on frozen Prithvi embeddings with 5-fold cross-validation.

In [ ]:
import torch.nn as nn
from sklearn.model_selection import KFold

# Feature Representation: [e_before, e_after, |e_after - e_before|]
diff_features = torch.abs(X_after - X_before)
multimodal_X = torch.cat([X_before, X_after, diff_features], dim=1) # (n_tiles, 768 * 3 = 2304)

class DeforestationLinearProbe(nn.Module):
    def __init__(self, in_dim=2304, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y_binary))
oof_probs = np.zeros(len(y_binary))

print("Training Supervised Linear Probe on Frozen Prithvi Embeddings (5-Fold CV)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(multimodal_X)):
    X_train, y_train = multimodal_X[train_idx].to(device), torch.tensor(y_binary[train_idx]).float().to(device)
    X_val, y_val = multimodal_X[val_idx].to(device), torch.tensor(y_binary[val_idx]).float().to(device)

    probe = DeforestationLinearProbe().to(device)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-2)
    criterion = nn.BCEWithLogitsLoss()

    probe.train()
    for epoch in range(40):
        optimizer.zero_grad()
        logits = probe(X_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()

    probe.eval()
    with torch.no_grad():
        val_logits = probe(X_val)
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
        oof_probs[val_idx] = val_probs
        oof_preds[val_idx] = (val_probs >= 0.5).astype(int)

rho_probe, p_probe = spearmanr(oof_probs, y_loss_frac)
prec_probe = precision_score(y_binary, oof_preds, zero_division=0)
rec_probe = recall_score(y_binary, oof_preds, zero_division=0)
iou_probe = jaccard_score(y_binary, oof_preds, zero_division=0)

print("\n=== EXPERIMENT 3: SUPERVISED LINEAR PROBE (CROSS-VALIDATED) ===")
print(f"Spearman Rho : {rho_probe:.3f} (p={p_probe:.4f})")
print(f"Precision    : {prec_probe:.3f}")
print(f"Recall       : {rec_probe:.3f}")
print(f"IoU (Jaccard): {iou_probe:.3f}")

### Cell 10: Comparative Summary Table

In [ ]:
import pandas as pd

summary_df = pd.DataFrame({
    "Method / Paradigm": [
        "1. Baseline: Unsupervised Cosine Distance",
        "2. Technique 2: dNDVI Gated + Otsu Dynamic Threshold",
        "3. Technique 1: Supervised Linear Probe (MLP Head)"
    ],
    "Spearman Rho": [round(rho_base, 3), round(rho_gated, 3), round(rho_probe, 3)],
    "Precision": [round(prec_base, 3), round(prec_gated, 3), round(prec_probe, 3)],
    "Recall": [round(rec_base, 3), round(rec_gated, 3), round(rec_probe, 3)],
    "IoU (Jaccard)": [round(iou_base, 3), round(iou_gated, 3), round(iou_probe, 3)],
    "IoU Gain vs Baseline": [
        "-",
        f"+{((iou_gated - iou_base) / (iou_base + 1e-6) * 100):.1f}%",
        f"+{((iou_probe - iou_base) / (iou_base + 1e-6) * 100):.1f}%"
    ]
})

print("\n===================== FINAL BENCHMARK COMPARISON TABLE =====================")
print(summary_df.to_string(index=False))

out_csv = "/content/drive/MyDrive/thesis_prithvi_poc/prithvi_optimization_results.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
summary_df.to_csv(out_csv, index=False)
print(f"\nSaved summary table to: {out_csv}")

### Cell 11: 5-Panel Publication Comparison Figure

In [ ]:
import matplotlib.pyplot as plt

# Reshape tile grids for visualization
grid_hansen = y_loss_frac.reshape(n_tiles_y, n_tiles_x)
grid_baseline = baseline_change.reshape(n_tiles_y, n_tiles_x)
grid_gated = gated_change.reshape(n_tiles_y, n_tiles_x)
grid_probe = oof_probs.reshape(n_tiles_y, n_tiles_x)

def s2_to_rgb(arr):
    # S2 bands: B2(0), B3(1), B4(2) -> RGB is (B4, B3, B2)
    rgb = np.stack([arr[2], arr[1], arr[0]], axis=-1)
    p2, p98 = np.percentile(rgb, (2, 98))
    return np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)

fig, axes = plt.subplots(1, 5, figsize=(28, 5.5))

axes[0].imshow(s2_to_rgb(before_arr))
axes[0].set_title("(a) Sentinel-2 BEFORE (2019)", fontsize=11, fontweight='bold')

axes[1].imshow(s2_to_rgb(after_arr))
axes[1].set_title("(b) Sentinel-2 AFTER (2024)", fontsize=11, fontweight='bold')

im2 = axes[2].imshow(grid_hansen, cmap="Reds", vmin=0, vmax=1)
axes[2].set_title("(c) Hansen Loss Ground Truth", fontsize=11, fontweight='bold')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

im3 = axes[3].imshow(grid_baseline, cmap="inferno")
axes[3].set_title(f"(d) Baseline Prithvi Cosine\n(IoU={iou_base:.2f}, Rho={rho_base:.2f})", fontsize=10)
plt.colorbar(im3, ax=axes[3], fraction=0.046)

im4 = axes[4].imshow(grid_probe, cmap="viridis", vmin=0, vmax=1)
axes[4].set_title(f"(e) Supervised Linear Probe\n(IoU={iou_probe:.2f}, Rho={rho_probe:.2f})", fontsize=10, fontweight='bold', color='darkgreen')
plt.colorbar(im4, ax=axes[4], fraction=0.046)

for ax in axes:
    ax.axis("off")

fig.suptitle("IOI / SYARIMO Mill: Prithvi-EO-2.0 Foundation Model Optimization Benchmark", fontsize=14, y=1.02)
plt.tight_layout()

out_fig = "/content/drive/MyDrive/thesis_prithvi_poc/prithvi_optimization_5panel.png"
plt.savefig(out_fig, dpi=250, bbox_inches="tight")
plt.show()
print(f"Publication-ready 5-panel figure saved to Drive: {out_fig}")